# GPU benchmark on a T4 (Colab)

Measures, with trained weights (gpt2 target, distilgpt2 draft):

- tokens/s and ms per token for greedy without cache, KV cache, batched and speculative decoding (k = 2, 4, 6), means with 95% bootstrap CIs over prompts and repeats;
- the measured speculative acceptance rate (greedy and temperature 1);
- continuous vs static batching throughput and latency on the same request trace.

**Runtime → Change runtime type → T4 GPU**, then run all cells (about 15-25 minutes). The last cell downloads `gpu_results.zip`. Unzip it in the repository root and commit:

```
results/gpu_t4.json  results/gpu_t4.md  results/gpu_t4_*.png
results/serving_gpu_t4.json  results/serving_gpu_t4.md
```

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
REPO = "https://github.com/LasyaRamachandruni/llm-inference-from-scratch.git"
NAME = "gpu_t4"   # output file stem; change if the GPU is not a T4

In [ ]:
import os, subprocess
if not os.path.exists("llm-inference-from-scratch"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir("llm-inference-from-scratch")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Colab already has a CUDA build of torch; install the rest without touching it
%pip install -q -e ".[hf,plots,serve]"
import torch
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 1) every decoding method on 32 chat prompts, 128 new tokens each, trained weights
!python -m infer.benchmark --pretrained --device cuda --target gpt2 --draft distilgpt2 \
    --prompts 32 --new-tokens 128 --ks 2 4 6 --name {NAME}

In [ ]:
# 2) continuous vs static batching on the GPU, same seeded trace for both
!python -m infer.serving.loadtest offline --pretrained --model gpt2 --device cuda \
    --requests 128 --repeats 3 --max-slots 32 --out results/serving_{NAME}.json

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f"results/{NAME}.md").read()))
display(Markdown(open(f"results/serving_{NAME}.md").read()))
import glob
for p in sorted(glob.glob(f"results/{NAME}_*.png")):
    display(Image(p, width=640))

In [ ]:
# 3) package the files to commit
import glob, zipfile
files = sorted(glob.glob(f"results/{NAME}.*") + glob.glob(f"results/{NAME}_*.png") + glob.glob(f"results/serving_{NAME}.*"))
with zipfile.ZipFile("gpu_results.zip", "w") as z:
    for f in files:
        z.write(f)
        print("added", f)
from google.colab import files as colab_files
colab_files.download("gpu_results.zip")

Then, in your clone:

```bash
unzip -o gpu_results.zip
git add results/ && git commit -m "Add T4 benchmark results" && git push
```

Paste the two markdown tables into the README in place of the "pending" rows.